# Look at mapping breadth and depth of 100 x metagenomes to singleclust genes

In [19]:
import polars as pl
import glob
import os
import screed
import csv

## Read in individual metag x species singleclust depth reports

These files contain depth-of-mapping information for all our genes.

In [2]:
DIR='../outputs.cds/singleclust/bam.bak'
template = '../outputs.cds/singleclust/bam/{metag}.x.{species}.depth.txt'

def read_depth(metag, species):
    filename = template.format(metag=metag, species=species)
    df = pl.read_csv(filename, separator='\t', has_header=False,
                 new_columns=('gene', 'pos', 'cov', 'foo'))
    sum_df = df.group_by('gene').agg(
        ((pl.col("cov") > 0).sum() / pl.col("pos").len()).alias("breadth")
    ).with_columns(
        (pl.lit(metag).alias("metag")),
        (pl.lit(species).alias("species"))
)
    return sum_df

read_depth('ERR1135199', 's__Cryptobacteroides sp900546925')

gene,breadth,metag,species
str,f64,str,str
"""FNIGJHLI_01182""",0.706865,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""GGBBOKIP_00385""",0.537143,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""FNIGJHLI_01271""",0.668721,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""GAHBKEGJ_00083""",0.927739,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""MFHJPMDA_01033""",0.989274,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
…,…,…,…
"""LHAJOBLP_00025""",0.933333,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""LELOEEPG_00745""",0.630983,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""FNIGJHLI_02652""",0.855763,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"


In [3]:
# Read them all in!
filenames = glob.glob(f"{DIR}/*.depth.txt")

dflist = []
for i, n in enumerate(filenames):
    if i % 100 == 0:
        print(f"{i} of {len(filenames)}")
    n = os.path.basename(n)
    metag, _, species, _ = n.split('.', 3)
    dflist.append(read_depth(metag, species))

depth_df = pl.concat(dflist)

print(f"read {len(filenames)} depth files.")

0 of 1400
100 of 1400
200 of 1400
300 of 1400
400 of 1400
500 of 1400
600 of 1400
700 of 1400
800 of 1400
900 of 1400
1000 of 1400
1100 of 1400
1200 of 1400
1300 of 1400
read 1400 depth files.


In [4]:
depth_df

gene,breadth,metag,species
str,f64,str,str
"""PHKCABJM_00446""",0.248313,"""ERR8314733""","""s__UBA2868 sp004552595"""
"""KBABMEIJ_01243""",0.763035,"""ERR8314733""","""s__UBA2868 sp004552595"""
"""HEOOOEGB_00722""",0.565058,"""ERR8314733""","""s__UBA2868 sp004552595"""
"""FACPLPEO_00440""",0.820323,"""ERR8314733""","""s__UBA2868 sp004552595"""
"""MPMJLGGA_01155""",0.927632,"""ERR8314733""","""s__UBA2868 sp004552595"""
…,…,…,…
"""LEHFNBLN_02301""",0.743333,"""SRR12795793""","""s__Bariatricus sp004560705"""
"""MDBJFMLD_01664""",0.359848,"""SRR12795793""","""s__Bariatricus sp004560705"""
"""NDCIOGGG_00214""",0.521484,"""SRR12795793""","""s__Bariatricus sp004560705"""


## Summarize our mapping depth results across all the metagenomes

In [5]:
# require 10% of each gene to be covered by at least one read
BREADTH_CUTOFF = 0.1

In [6]:
# aggregate across all metagenomes;
# calculate fraction of metagenomes for which gene mapping exceeds our breadth cutoff
agg_df = depth_df.group_by(['species', 'gene']).agg(
    ((pl.col("breadth") >= BREADTH_CUTOFF).sum() / pl.col("breadth").len()).alias("f")
)
agg_df

species,gene,f
str,str,f64
"""s__Prevotella sp002251295""","""GDBLKLPM_00824""",0.83
"""s__Prevotella sp000434975""","""JIFGGGBB_01218""",0.86
"""s__Prevotella sp002251295""","""EFBIIPPC_01725""",0.82
"""s__Sodaliphilus sp004557565""","""GFGDBLKC_00292""",0.89
"""s__Sodaliphilus sp004557565""","""KANCODDC_00205""",0.91
…,…,…
"""s__UBA2868 sp004552595""","""PCFNEHJA_01076""",0.62
"""s__Holdemanella porci""","""CAJDFOMO_00625""",0.76
"""s__Cryptobacteroides sp9005469…","""FNIGJHLI_01255""",0.81


In [21]:
# print information out by species
for species in sorted(agg_df['species'].unique()):
    print(species)
    foo_df = agg_df.filter(pl.col("species") == species)
    foo_df = foo_df.sort(by='f', descending=True).filter(pl.col('f') > 0.8)
    print(foo_df)

    top50_names = set(foo_df.head(50)['gene'].to_list())

    outfile = f'../outputs.cds/singleclust/{species}.cds.min50.dedup.top50.fa'
    print(outfile)
    outfp = open(outfile, 'wt')
    for record in screed.open(f'../outputs.cds/singleclust/{species}.cds3.min50.dedup.fa'):
        name = record.name.split(' ')[0]
        if name in top50_names:
            top50_names.remove(name)
            outfp.write(f'>{record.name}\n{record.sequence}\n')
    assert not top50_names
    outfp.close()

    outfile2 = f'../outputs.cds/singleclust/{species}.cds.min50.dedup.top50.csv'
    print(outfile2)
    outfp = open(outfile2, 'w', newline='')
    w = csv.writer(outfp)

    for name in foo_df.head(50)['gene'].to_list():
        w.writerow(['0', species, name, "(not reviewed)"])
    outfp.close()
            
    

s__Bariatricus sp004560705
shape: (15, 3)
┌────────────────────────────┬────────────────┬──────┐
│ species                    ┆ gene           ┆ f    │
│ ---                        ┆ ---            ┆ ---  │
│ str                        ┆ str            ┆ f64  │
╞════════════════════════════╪════════════════╪══════╡
│ s__Bariatricus sp004560705 ┆ CHOLCMOC_00863 ┆ 0.97 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_00701 ┆ 0.95 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_02301 ┆ 0.93 │
│ s__Bariatricus sp004560705 ┆ FMMMFAAO_00746 ┆ 0.89 │
│ s__Bariatricus sp004560705 ┆ NDCIOGGG_00214 ┆ 0.89 │
│ …                          ┆ …              ┆ …    │
│ s__Bariatricus sp004560705 ┆ NGBMPMHL_00028 ┆ 0.84 │
│ s__Bariatricus sp004560705 ┆ NOACAEKI_00860 ┆ 0.83 │
│ s__Bariatricus sp004560705 ┆ LNGHPPMH_00679 ┆ 0.83 │
│ s__Bariatricus sp004560705 ┆ EMNHIHKA_01693 ┆ 0.82 │
│ s__Bariatricus sp004560705 ┆ ELOEBFJM_01459 ┆ 0.81 │
└────────────────────────────┴────────────────┴──────┘
../outputs.cds/singlecl

## Explore specific gene/species/metag combinations

In [8]:
depth_df.filter((pl.col("gene") == 'EHOAPHDI_01174')).sort(by='breadth')

gene,breadth,metag,species
str,f64,str,str


In [9]:
depth_df.filter((pl.col("gene") == 'EHOAPHDI_01174')).sort(by='breadth').filter(pl.col("metag") == "ERR1135199")

gene,breadth,metag,species
str,f64,str,str
